## Preprocessing, models and comparison

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.db import get_engine, fetch_table
from src.config import FEATURE_COLS_NO_WIFI
from src.config import MODEL_TEXT_COLS, MODEL_NUMBER_COLS, MODEL_RATING_COLS, MODEL_RATING_COLS_NO_WIFI
from src.features import make_features

print("Project root:", project_root)

Project root: d:\OMEN-16\ML-Projects\airline-satisfaction-ml


In [2]:
engine = get_engine()
df = fetch_table(engine)

X = make_features(df)
y = (df["satisfaction"] == "Satisfied").astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

X_train_no_wifi = X_train[FEATURE_COLS_NO_WIFI]

print("X_train:", X_train.shape)
print("X_train_no_wifi:", X_train_no_wifi.shape)
print("Share satisfied in y_train:", round(y_train.mean(), 4))

X_train: (103904, 18)
X_train_no_wifi: (103904, 17)
Share satisfied in y_train: 0.4345


#### Preprocessing for tree based models

In [3]:
tree_prep = ColumnTransformer(
    [
        ("class_order", OrdinalEncoder(categories=[["Economy", "Economy Plus", "Business"]]), ["travel_class"]),
        ("two_labels", OrdinalEncoder(), ["customer_type", "type_of_travel"]),
    ],
    remainder="passthrough",
    verbose_feature_names_out=False,
)
tree_prep.set_output(transform="pandas")

print("tree_prep is built, nothing is learned yet")

tree_prep is built, nothing is learned yet


#### fit on traning rows

In [4]:
X_train_tree = tree_prep.fit_transform(X_train)

print("Shape:", X_train_tree.shape)
print("Missing values:", X_train_tree.isnull().sum().sum())
print()
print(X_train_tree.dtypes.value_counts())
print()
print(list(X_train_tree.columns))

Shape: (103904, 18)
Missing values: 0

int64      15
float64     3
Name: count, dtype: int64

['travel_class', 'customer_type', 'type_of_travel', 'age', 'flight_distance', 'departure_delay', 'ease_of_online_booking', 'checkin_service', 'online_boarding', 'onboard_service', 'seat_comfort', 'leg_room_service', 'cleanliness', 'food_and_drink', 'inflight_service', 'inflight_wifi_service', 'inflight_entertainment', 'baggage_handling']


In [5]:
X_train_tree.head()

,travel_class,customer_type,type_of_travel,age,flight_distance,departure_delay,ease_of_online_booking,checkin_service,online_boarding,onboard_service,seat_comfort,leg_room_service,cleanliness,food_and_drink,inflight_service,inflight_wifi_service,inflight_entertainment,baggage_handling
57644,2.0,1.0,0.0,56,1848,0,4,3,2,4,5,4,1,4,4,4,4,4
103332,0.0,1.0,1.0,70,1371,3,3,3,4,1,5,3,4,4,1,3,1,1
76605,2.0,1.0,0.0,48,2473,0,1,5,4,5,5,5,5,4,5,3,5,5
121595,0.0,1.0,1.0,63,912,40,3,2,3,4,5,4,5,5,2,3,5,2
107087,2.0,1.0,0.0,57,562,12,4,3,4,4,4,4,3,4,4,4,4,4


#### checking how the text turn into numbers

In [6]:
# how did the text turn into numbers?
print(pd.crosstab(X_train["travel_class"], X_train_tree["travel_class"]))
print()
print()
print(pd.crosstab(X_train["customer_type"], X_train_tree["customer_type"]))
print()
print()
print(pd.crosstab(X_train["type_of_travel"], X_train_tree["type_of_travel"]))

travel_class    0.0   1.0    2.0
travel_class                    
Business          0     0  49699
Economy       46612     0      0
Economy Plus      0  7593      0


customer_type    0.0    1.0
customer_type              
First-time     19107      0
Returning          0  84797


type_of_travel    0.0    1.0
type_of_travel              
Business        71725      0
Personal            0  32179


#### Preprocessing for linear models

In [7]:
linear_prep = ColumnTransformer(
    [
        ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False), MODEL_TEXT_COLS + MODEL_RATING_COLS),
        ("scale", StandardScaler(), MODEL_NUMBER_COLS),
    ],
    verbose_feature_names_out=False,
)
linear_prep.set_output(transform="pandas")


# this is the same thing for the version without wifi
linear_prep_no_wifi = ColumnTransformer(
    [
        ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False), MODEL_TEXT_COLS + MODEL_RATING_COLS_NO_WIFI),
        ("scale", StandardScaler(), MODEL_NUMBER_COLS),
    ],
    verbose_feature_names_out=False,
)
linear_prep_no_wifi.set_output(transform="pandas")

print("linear_prep and linear_prep_no_wifi are built, nothing is learned yet")

linear_prep and linear_prep_no_wifi are built, nothing is learned yet


#### fit on the training

In [8]:
X_train_lin = linear_prep.fit_transform(X_train)

print("Shape:", X_train_lin.shape)
print("Missing values:", X_train_lin.isnull().sum().sum())
print()
print("First 10 column names:")
print(list(X_train_lin.columns[:10]))
print()
print("Last 5 column names:")
print(list(X_train_lin.columns[-5:]))

Shape: (103904, 79)
Missing values: 0

First 10 column names:
['customer_type_First-time', 'customer_type_Returning', 'type_of_travel_Business', 'type_of_travel_Personal', 'travel_class_Business', 'travel_class_Economy', 'travel_class_Economy Plus', 'ease_of_online_booking_0', 'ease_of_online_booking_1', 'ease_of_online_booking_2']

Last 5 column names:
['baggage_handling_4', 'baggage_handling_5', 'age', 'flight_distance', 'departure_delay']


In [9]:
X_train_lin.head()

,customer_type_First-time,customer_type_Returning,type_of_travel_Business,type_of_travel_Personal,travel_class_Business,travel_class_Economy,travel_class_Economy Plus,ease_of_online_booking_0,ease_of_online_booking_1,ease_of_online_booking_2,...,inflight_entertainment_4,inflight_entertainment_5,baggage_handling_1,baggage_handling_2,baggage_handling_3,baggage_handling_4,baggage_handling_5,age,flight_distance,departure_delay
57644,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.095776,0.658793,-0.385030
103332,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,2.019955,0.180855,-0.306583
76605,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.567674,1.285022,-0.385030
121595,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.557866,-0.279047,0.660925
107087,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.161789,-0.629735,-0.071244


#### cheking what did the scaler learn

In [10]:

scaler = linear_prep.named_transformers_["scale"]

learned = pd.Series(scaler.mean_, index=MODEL_NUMBER_COLS)
all_data = X[MODEL_NUMBER_COLS].mean()

print("Average the scaler learned from X_train:")
print(learned.round(2))
print()
print("Average of the same columns in all the data:")
print(all_data.round(2))

Average the scaler learned from X_train:
age                  39.40
flight_distance    1190.50
departure_delay      14.72
dtype: float64

Average of the same columns in all the data:
age                  39.43
flight_distance    1190.32
departure_delay      14.71
dtype: float64


observation:

- here the two sets of averages are very close but not identical, which proves the scaler learned only from the training rows. If it had used all the data, the two would match exactly.

#### checking what did the encoder learn? 

In [11]:
encoder = linear_prep.named_transformers_["encode"]
encoded_cols = MODEL_TEXT_COLS + MODEL_RATING_COLS

for col_name, labels in zip(encoded_cols, encoder.categories_):
    print(col_name.ljust(26), labels.tolist())

customer_type              ['First-time', 'Returning']
type_of_travel             ['Business', 'Personal']
travel_class               ['Business', 'Economy', 'Economy Plus']
ease_of_online_booking     [0, 1, 2, 3, 4, 5]
checkin_service            [1, 2, 3, 4, 5]
online_boarding            [0, 1, 2, 3, 4, 5]
onboard_service            [0, 1, 2, 3, 4, 5]
seat_comfort               [1, 2, 3, 4, 5]
leg_room_service           [0, 1, 2, 3, 4, 5]
cleanliness                [0, 1, 2, 3, 4, 5]
food_and_drink             [0, 1, 2, 3, 4, 5]
inflight_service           [0, 1, 2, 3, 4, 5]
inflight_wifi_service      [0, 1, 2, 3, 4, 5]
inflight_entertainment     [0, 1, 2, 3, 4, 5]
baggage_handling           [1, 2, 3, 4, 5]


#### Transform

In [12]:
# transform only applies what was learned, it learns nothing new
few_rows = X_train.head()

again = linear_prep.transform(few_rows)
first_rows = X_train_lin.head()

print("Same numbers as before:", (again.values == first_rows.values).all())

Same numbers as before: True


#### one Made up passenger, the way streamlit app will send it later

In [13]:
# one made-up passenger, the way the Streamlit app will send it later
# Actually this is the same we have made in the featuring engineering file
one_passenger = pd.DataFrame([{
    "customer_type": "Returning", "type_of_travel": "Business", "travel_class": "Business",
    "age": 40, "flight_distance": 1500, "departure_delay": 10,
    "ease_of_online_booking": 4, "checkin_service": 4, "online_boarding": 4,
    "onboard_service": 4, "seat_comfort": 4, "leg_room_service": 4,
    "cleanliness": 4, "food_and_drink": 4, "inflight_service": 4,
    "inflight_wifi_service": 3, "inflight_entertainment": 4, "baggage_handling": 4,
}])

one_tree = tree_prep.transform(one_passenger)
one_lin = linear_prep.transform(one_passenger)

print("Tree version  :", one_tree.shape)
print("Linear version:", one_lin.shape)
one_tree.T

Tree version  : (1, 18)
Linear version: (1, 79)


,0
travel_class,2.0
customer_type,1.0
type_of_travel,0.0
age,40.0
flight_distance,1500.0
departure_delay,10.0
ease_of_online_booking,4.0
checkin_service,4.0
online_boarding,4.0
onboard_service,4.0


#### checking whcich linear column are we on

In [14]:
one_lin_row = one_lin.T
one_lin_row[one_lin_row[0] != 0]

,0
customer_type_Returning,1.000000
type_of_travel_Business,1.000000
travel_class_Business,1.000000
ease_of_online_booking_4,1.000000
checkin_service_4,1.000000
online_boarding_4,1.000000
onboard_service_4,1.000000
seat_comfort_4,1.000000
leg_room_service_4,1.000000
cleanliness_4,1.000000


In [15]:
# a score the training rows never had: baggage_handling = 0
odd_passenger = one_passenger.copy()
odd_passenger["baggage_handling"] = 0

odd_lin = linear_prep.transform(odd_passenger)

baggage_columns = odd_lin.columns.str.startswith("baggage_handling")
print("Baggage columns for this passenger:")
print(odd_lin.loc[:, baggage_columns])

Baggage columns for this passenger:
   baggage_handling_1  baggage_handling_2  baggage_handling_3  \
0                 0.0                 0.0                 0.0   

   baggage_handling_4  baggage_handling_5  
0                 0.0                 0.0  


Note :

- these All five baggage columns are 0.0, because the encoder doesn't know a score of 0 and, thanks to handle_unknown="ignore", leaves it at all zeros.

## without wifi versions

In [16]:
X_train_tree_no_wifi = tree_prep.fit_transform(X_train_no_wifi)
X_train_lin_no_wifi = linear_prep_no_wifi.fit_transform(X_train_no_wifi)

print("Tree, without wifi  :", X_train_tree_no_wifi.shape)
print("Linear, without wifi:", X_train_lin_no_wifi.shape)
print()
print("Wifi columns left (tree)  :", X_train_tree_no_wifi.columns.str.contains("wifi").sum())
print("Wifi columns left (linear):", X_train_lin_no_wifi.columns.str.contains("wifi").sum())

Tree, without wifi  : (103904, 17)
Linear, without wifi: (103904, 73)

Wifi columns left (tree)  : 0
Wifi columns left (linear): 0


#### the four column counts

In [17]:
print("Columns: tree / tree without wifi:", X_train_tree.shape[1], "/", X_train_tree_no_wifi.shape[1])
print("Columns: linear / linear without wifi:", X_train_lin.shape[1], "/", X_train_lin_no_wifi.shape[1])

Columns: tree / tree without wifi: 18 / 17
Columns: linear / linear without wifi: 79 / 73


In [18]:
import time

from sklearn.base import clone
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

print("All the model tools are imported")

All the model tools are imported


In [19]:
# each pipeline = a preprocessor + a model. clone() gives every model its own copy of the preprocessor.
log_reg = make_pipeline(clone(linear_prep), LogisticRegression(max_iter=2000))
knn = make_pipeline(clone(linear_prep), KNeighborsClassifier())
tree = make_pipeline(clone(tree_prep), DecisionTreeClassifier(random_state=42))
forest = make_pipeline(clone(tree_prep), RandomForestClassifier(random_state=42, n_jobs=-1))
grad_boost = make_pipeline(clone(tree_prep), GradientBoostingClassifier(random_state=42))
xgb = make_pipeline(clone(tree_prep), XGBClassifier(random_state=42, n_jobs=-1))
lgbm = make_pipeline(clone(tree_prep), LGBMClassifier(random_state=42, n_jobs=-1, verbose=-1))

models = {
    "Logistic Regression": log_reg,
    "KNN": knn,
    "Decision Tree": tree,
    "Random Forest": forest,
    "Gradient Boosting": grad_boost,
    "XGBoost": xgb,
    "LightGBM": lgbm,
}

print(len(models), "pipelines built")

7 pipelines built


In [20]:
for name in models:
    print(name.ljust(22), list(models[name].named_steps))

Logistic Regression    ['columntransformer', 'logisticregression']
KNN                    ['columntransformer', 'kneighborsclassifier']
Decision Tree          ['columntransformer', 'decisiontreeclassifier']
Random Forest          ['columntransformer', 'randomforestclassifier']
Gradient Boosting      ['columntransformer', 'gradientboostingclassifier']
XGBoost                ['columntransformer', 'xgbclassifier']
LightGBM               ['columntransformer', 'lgbmclassifier']


#### training with wifif

In [21]:
train_times = {}

for name in models:
    start = time.time()
    models[name].fit(X_train, y_train)
    seconds = time.time() - start

    train_times[name] = round(seconds, 1)
    print(name.ljust(22), round(seconds, 1), "seconds")

Logistic Regression    0.8 seconds
KNN                    0.2 seconds
Decision Tree          0.4 seconds
Random Forest          1.1 seconds
Gradient Boosting      10.4 seconds
XGBoost                2.1 seconds
LightGBM               0.5 seconds


#### Building and training without wifi

In [22]:
log_reg_nw = make_pipeline(clone(linear_prep_no_wifi), LogisticRegression(max_iter=2000))
knn_nw = make_pipeline(clone(linear_prep_no_wifi), KNeighborsClassifier())
tree_nw = make_pipeline(clone(tree_prep), DecisionTreeClassifier(random_state=42))
forest_nw = make_pipeline(clone(tree_prep), RandomForestClassifier(random_state=42, n_jobs=-1))
grad_boost_nw = make_pipeline(clone(tree_prep), GradientBoostingClassifier(random_state=42))
xgb_nw = make_pipeline(clone(tree_prep), XGBClassifier(random_state=42, n_jobs=-1))
lgbm_nw = make_pipeline(clone(tree_prep), LGBMClassifier(random_state=42, n_jobs=-1, verbose=-1))

models_no_wifi = {
    "Logistic Regression": log_reg_nw,
    "KNN": knn_nw,
    "Decision Tree": tree_nw,
    "Random Forest": forest_nw,
    "Gradient Boosting": grad_boost_nw,
    "XGBoost": xgb_nw,
    "LightGBM": lgbm_nw,
}

print(len(models_no_wifi), "pipelines built")

7 pipelines built


In [23]:
train_times_no_wifi = {}

for name in models_no_wifi:
    start = time.time()
    models_no_wifi[name].fit(X_train_no_wifi, y_train)
    seconds = time.time() - start

    train_times_no_wifi[name] = round(seconds, 1)
    print(name.ljust(22), round(seconds, 1), "seconds")

Logistic Regression    0.7 seconds
KNN                    0.2 seconds
Decision Tree          0.5 seconds
Random Forest          1.2 seconds
Gradient Boosting      16.9 seconds
XGBoost                0.7 seconds
LightGBM               0.9 seconds


#### Checking what did each model build

In [24]:
print("Logistic Regression coefficients:", models["Logistic Regression"][-1].coef_.shape)
print("KNN neighbours               :", models["KNN"][-1].n_neighbors)
print("Decision Tree depth          :", models["Decision Tree"][-1].get_depth())
print("Decision Tree leaves         :", models["Decision Tree"][-1].get_n_leaves())
print("Random Forest trees          :", len(models["Random Forest"][-1].estimators_))
print("Gradient Boosting trees      :", models["Gradient Boosting"][-1].n_estimators_)
print("XGBoost trees                :", models["XGBoost"][-1].get_booster().num_boosted_rounds())
print("LightGBM trees               :", models["LightGBM"][-1].n_estimators_)

Logistic Regression coefficients: (1, 79)
KNN neighbours               : 5
Decision Tree depth          : 41
Decision Tree leaves         : 4343
Random Forest trees          : 100
Gradient Boosting trees      : 100
XGBoost trees                : 100
LightGBM trees               : 100


#### Checking do they work on raw data?

In [25]:
print("Actual answers for the first 5 training passengers:", y_train.head().tolist())
print()

for name in models:
    print(name.ljust(22), models[name].predict(X_train.head()).tolist())

Actual answers for the first 5 training passengers: [1, 0, 1, 0, 1]

Logistic Regression    [1, 0, 1, 0, 1]
KNN                    [1, 0, 1, 0, 1]
Decision Tree          [1, 0, 1, 0, 1]
Random Forest          [1, 0, 1, 0, 1]
Gradient Boosting      [1, 0, 1, 0, 1]
XGBoost                [1, 0, 1, 0, 1]
LightGBM               [1, 0, 1, 0, 1]


In [26]:
print("Chance of satisfied for the made-up passenger:")

for name in models:
    chance = models[name].predict_proba(one_passenger)[0, 1]
    print(name.ljust(22), round(chance * 100, 1), "%")

Chance of satisfied for the made-up passenger:
Logistic Regression    87.4 %
KNN                    100.0 %
Decision Tree          100.0 %
Random Forest          100.0 %
Gradient Boosting      93.5 %
XGBoost                98.6 %
LightGBM               96.9 %


In [27]:
# accuracy on 5,000 of the training rows the models learned from
sample_X = X_train.head(5000)
sample_y = y_train.head(5000)

for name in models:
    accuracy = models[name].score(sample_X, sample_y)
    print(name.ljust(22), round(accuracy * 100, 2), "%")

Logistic Regression    93.92 %
KNN                    95.94 %
Decision Tree          100.0 %
Random Forest          100.0 %
Gradient Boosting      94.5 %
XGBoost                97.48 %
LightGBM               96.82 %


In [28]:
times = pd.DataFrame({"with wifi (seconds)": train_times,
                      "without wifi (seconds)": train_times_no_wifi})
times

,with wifi (seconds),without wifi (seconds)
Logistic Regression,0.8,0.7
KNN,0.2,0.2
Decision Tree,0.4,0.5
Random Forest,1.1,1.2
Gradient Boosting,10.4,16.9
XGBoost,2.1,0.7
LightGBM,0.5,0.9
